In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/sonar.csv'
df = pd.read_csv(url, header=None)
print('Dataset Shape:', df.shape)
df.head()

Dataset Shape: (208, 61)


,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R


In [2]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1].map({'R': 0, 'M': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print('Test Accuracy:', accuracy)

Test Accuracy: 0.8333333333333334


In [3]:
cv_scores = cross_val_score(
    model, scaler.fit_transform(X), y, cv=5, scoring='accuracy'
)
print('5-Fold Cross-Validation Scores:')
print(cv_scores)
print('Mean CV Accuracy:', cv_scores.mean())

5-Fold Cross-Validation Scores:
[0.4047619  0.69047619 0.73809524 0.75609756 0.58536585]
Mean CV Accuracy: 0.634959349593496


In [4]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [100, 200, 500]
}

grid_search = GridSearchCV(
    LogisticRegression(random_state=42), param_grid,
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)
print('Best Parameters:')
print(grid_search.best_params_)
print('\nBest Cross-Validation Accuracy:', grid_search.best_score_)

Best Parameters:
{'C': 0.01, 'max_iter': 100, 'solver': 'lbfgs'}

Best Cross-Validation Accuracy: 0.7777183600713012


In [5]:
best_model = grid_search.best_estimator_
y_pred_grid = best_model.predict(X_test_scaled)
grid_accuracy = accuracy_score(y_test, y_pred_grid)
print('Optimized Model Test Accuracy:', grid_accuracy)

Optimized Model Test Accuracy: 0.8095238095238095


In [6]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42))
])

pipeline.fit(X_train, y_train)
pipeline_pred = pipeline.predict(X_test)
pipeline_accuracy = accuracy_score(y_test, pipeline_pred)
print('Pipeline Test Accuracy:', pipeline_accuracy)

pipeline_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print('\nPipeline 5-Fold CV Scores:')
print(pipeline_scores)
print('Pipeline Mean Accuracy:', pipeline_scores.mean())

Pipeline Test Accuracy: 0.8333333333333334

Pipeline 5-Fold CV Scores:
[0.4047619  0.69047619 0.73809524 0.75609756 0.58536585]
Pipeline Mean Accuracy: 0.634959349593496


In [7]:
print('===== FINAL PERFORMANCE COMPARISON =====')
print('Train-Test Split Accuracy:', round(accuracy * 100, 2), '%')
print('K-Fold Mean Accuracy:', round(cv_scores.mean() * 100, 2), '%')
print('Grid Search Test Accuracy:', round(grid_accuracy * 100, 2), '%')
print('Pipeline Mean Accuracy:', round(pipeline_scores.mean() * 100, 2), '%')

===== FINAL PERFORMANCE COMPARISON =====
Train-Test Split Accuracy: 83.33 %
K-Fold Mean Accuracy: 63.5 %
Grid Search Test Accuracy: 80.95 %
Pipeline Mean Accuracy: 63.5 %
